# SuperBook Local Python Video Engine • Wan2.1 VACE 1.3B

Zero-cost GPU proof for the **same Python backend** that can later run on a local GPU machine or a hosted GPU. It does not modify the SuperBook Flutter app.

Pipeline: scene image → semantic motion prompt → Wan2.1 VACE 1.3B → MP4.

In [ ]:
!nvidia-smi
!python -m pip install -q -U diffusers transformers accelerate safetensors pillow imageio[ffmpeg] ipywidgets

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Kaggle: Notebook options → Accelerator → GPU.')

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display
from pathlib import Path

uploader = FileUpload(accept='image/*', multiple=False, description='Upload scene image')
display(uploader)
print('Upload the actual SuperBook scene image, then run the next cell.')

In [ ]:
from pathlib import Path
from PIL import Image

if not uploader.value:
    raise RuntimeError('No image uploaded yet.')
item = next(iter(uploader.value.values())) if isinstance(uploader.value, dict) else uploader.value[0]
name = item['name'] if isinstance(item, dict) else item.name
data = item['content'] if isinstance(item, dict) else item.content
IMAGE_PATH = Path('/kaggle/working') / name
IMAGE_PATH.write_bytes(bytes(data))
image = Image.open(IMAGE_PATH).convert('RGB')
print('Image:', IMAGE_PATH, image.size)
display(image.resize((min(768, image.width), int(image.height * min(768, image.width) / image.width))))

In [ ]:
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

MODEL_ID = 'Wan-AI/Wan2.1-VACE-1.3B'
print('Loading', MODEL_ID, 'with CPU offload...')
pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
print('Model loaded')

In [ ]:
MOTION_PROMPT = '''A cinematic storybook scene. The main character moves naturally through the existing scene: takes two slow steps toward the table, turns toward the other character, raises one hand while speaking, and naturally shifts body weight. Subtle breathing and clothing movement. The other character listens with small natural head and posture movement. Preserve the exact characters, faces, clothing, room, lighting, furniture and composition from the input image. No new characters, no scene change, no camera cut, no text, no morphing, no duplicated limbs.'''
print(MOTION_PROMPT)

In [ ]:
generator = torch.Generator(device='cuda').manual_seed(42)
result = pipe(
    image=image,
    prompt=MOTION_PROMPT,
    num_frames=49,
    num_inference_steps=20,
    generator=generator,
)
frames = result.frames[0]
OUT = '/kaggle/working/superbook_wan_vace.mp4'
export_to_video(frames, OUT, fps=16)
print('Generated:', OUT)

In [ ]:
from IPython.display import Video, display
display(Video('/kaggle/working/superbook_wan_vace.mp4', embed=True))

## Decision gate

**PASS:** meaningful character/body motion with preserved identity and scene continuity.

**FAIL:** camera drift only, severe face/limb corruption, scene replacement, or negligible motion.

If it passes, the next step is connecting the isolated Python HTTP engine to SuperBook. If it fails, the HTTP contract stays and only the model backend changes.